# 🌿 AgroAI: Multi-Dataset Plant Pathology Deep Learning Training Pipeline

This end-to-end Google Colab notebook aggregates and trains deep vision backbones (MobileNetV3 / EfficientNet) across **67 disease classes across 22 crops** using a multi-dataset blend:
- **PlantVillage** (54,303 lab benchmark images)
- **PlantDoc** (In-field farm photos in direct sunlight with background clutter)
- **Paddy Doctor & Cassava** (Real-world smallholder farm captures)
- **Cash Crops** (Sugarcane, Cotton, Coffee, Wheat Rust, Banana)

### ⚡ Hardware Accelerator:
Ensure GPU is enabled: **Runtime > Change runtime type > T4 GPU**.

In [ ]:
# 1. Check GPU Compute Status
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Model:       {torch.cuda.get_device_name(0)}")
    device = torch.device("cuda")
else:
    print("Running on CPU. For fast training, enable T4 GPU in Runtime settings.")
    device = torch.device("cpu")

In [ ]:
# 2. Clone AgroAI Codebase
!git clone https://github.com/Manmeet-Singh-Deol/AI-Crop-Predictor.git
%cd AI-Crop-Predictor
!pip install -q timm albumentations opencv-python-headless pillow reportlab

In [ ]:
# 3. Download Open-Access Datasets (PlantVillage + PlantDoc In-Field Mirrors)
import os

print("📥 Downloading PlantVillage Benchmark Dataset...")
!wget -q -O plantvillage.zip https://github.com/spMohanty/PlantVillage-Dataset/archive/refs/heads/master.zip
!unzip -q plantvillage.zip -d /content/raw_plantvillage

print("📥 Downloading PlantDoc In-The-Wild Farm Dataset...")
!wget -q -O plantdoc.zip https://github.com/pratikkayal/PlantDoc-Dataset/archive/refs/heads/master.zip
!unzip -q plantdoc.zip -d /content/raw_plantdoc

print("✅ Core datasets downloaded successfully!")

In [ ]:
# 4. (Optional) Ingest Kaggle Field Datasets (PaddyDoctor, Sugarcane, Cassava, RoCoLe)
# To download Kaggle datasets, upload your kaggle.json or uncomment below:
"""
!mkdir -p ~/.kaggle
!cp /content/kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!kaggle competitions download -c paddy-disease-classification -p /content/raw_paddy --unzip
!kaggle datasets download -d nirmalsankalana/sugarcane-leaf-disease-dataset -p /content/raw_sugarcane --unzip
!kaggle competitions download -c cassava-leaf-disease-classification -p /content/raw_cassava --unzip
"""
print("Dataset sources ready for consolidation.")

In [ ]:
# 5. Aggregate & Consolidate All Datasets into Unified 67-Class Taxonomy
from backend.dataset_aggregator import merge_datasets_into_unified_structure

sources = [
    "/content/raw_plantvillage/PlantVillage-Dataset-master/raw/color",
    "/content/raw_plantdoc/PlantDoc-Dataset-master/train",
    "/content/raw_plantdoc/PlantDoc-Dataset-master/test",
    "/content/raw_paddy",
    "/content/raw_sugarcane",
    "/content/raw_cassava"
]

output_dir = "/content/unified_dataset"
stats = merge_datasets_into_unified_structure(sources, output_dir, train_ratio=0.85)
print(f"Unified dataset constructed at: {output_dir}")

In [ ]:
# 6. Launch Deep Learning Training Pipeline with In-Field Data Augmentation
from train import train_model

weights_path = train_model(
    data_dir="/content/unified_dataset",
    epochs=12,
    batch_size=64 if torch.cuda.is_available() else 16,
    learning_rate=1e-3,
    output_weights_path="backend/model_weights.pth",
    device_name="auto",
    label_smoothing=0.1
)
print(f"Training finished! Model saved to: {weights_path}")

In [ ]:
# 7. Verify Grad-CAM Visual Explainability on Sample Images
import cv2
import numpy as np
from PIL import Image
from backend.classifier import get_inference_engine
from backend.gradcam import generate_visual_explanation
from backend.sample_images import generate_sugarcane_red_rot, generate_wheat_yellow_rust

test_leaf = generate_wheat_yellow_rust()
engine = get_inference_engine()
preds, features = engine.predict(test_leaf)
cam_b64, heatmap_b64, overlay_b64 = generate_visual_explanation(test_leaf, preds[0]['crop_disease'])

print(f"Top Prediction: {preds[0]['crop']} - {preds[0]['disease']} ({preds[0]['confidence']}% confidence)")
print("Grad-CAM Heatmap generation verified!")

In [ ]:
# 8. Download Trained Weights to Your Local Machine
from google.colab import files
files.download('backend/model_weights.pth')
print("Model weights downloading to your computer!")